# 🏗️ Notebook: Introduction to Structured Outputs & Validation with LLMs

In this notebook you will learn how to use Large Language Models (LLMs) to generate structured outputs.

## 📚 Sources

- [OpenAI: Validating LLM Outputs](https://platform.openai.com/docs/guides/structured-outputs?api-mode=chat)
- [Pydantic Documentation](https://pydantic.dev/)
- [Ollama Structured Outputs](https://ollama.com/blog/structured-outputs)

---

Good luck with experimenting and validating! 🤗

In [1]:
# Import the required libraries
from pydantic import BaseModel, Field
from openai import OpenAI
from typing import List, Literal, Optional
import json

In [2]:
LLM_URL = "http://132.199.138.16:11434/v1"
LLM_REASONING = "gemma4:26b"  # the reasoning MoE model from the previous notebook

# Ollama's native API (used later for regex-constrained output) lives at the same
# host, just without the "/v1" suffix used by the OpenAI-compatible API.
OLLAMA_HOST = LLM_URL.removesuffix("/v1")

In [3]:
client = OpenAI(
    base_url=LLM_URL,
    api_key="ollama",
)

As in the previous notebook, we'll use `LLM_REASONING` (`gemma4:26b`) throughout — accessed via Ollama's OpenAI-compatible API. **Reminder**: besides Ollama, there are other providers that offer OpenAI-compatible APIs, e.g., [vLLM](https://docs.vllm.ai/en/v0.8.2/features/structured_outputs.html).

**Structured outputs** enable restricting the output of a model to a specific format defined by a **JSON schema**. Ollama currently supports structured outputs for JSON format, as do OpenAI's endpoints for their GPT models. The vLLM library, which also allows us to use open-source LLMs, supports regex as well, although there are still some bugs (as of October 2025).

Two things we'll do consistently from here on:

- **Disable thinking for simple extractions.** `LLM_REASONING` thinks by default, but for the small, simple schemas in the examples below, that extra reasoning step is just overhead — it costs time without meaningfully improving the result. We'll turn it off with `reasoning_effort="none"`. This is optional, not a rule: for harder tasks (like the multi-field document extraction later in this notebook), it can genuinely help, so we'll leave it enabled there.
- **Always include the JSON schema in the prompt.** Even though `response_format` already forces the output to match the schema structurally, the model still has to guess what each field *means* unless we tell it. So we'll pass `model.model_json_schema()` into the prompt text itself, not just as `response_format` — that way the model isn't left guessing in the dark about what a field name refers to.

Use cases for structured outputs:

- Extract data from documents
- Extract data from images
- Structure all language model responses

## Interlude: JSON – JavaScript Object Notation

**JSON** is a simple, text-based format for storing and exchanging data.  
It is supported by many programming languages and is especially popular for web APIs.

### Basic Principles
- Data is stored as **key-value pairs**.
- Structures can be **nested** (objects within objects, lists within objects).

### Data Types in JSON
- **String**: Text in quotes, e.g., `"Hello"`
- **Number**: Integers or decimals, e.g., `42` or `3.14`
- **Boolean**: Truth values `true` or `false`
- **Null**: Empty value `null`
- **Array**: List of values, e.g., `[1, 2, 3]`. Different data types are also possible, e.g., `["Text", 42, true]`
- **Object**: Collection of key-value pairs, e.g., `{"name": "Max", "age": 30}`

### Example
```json
{
  "name": "Bello",
  "tier": "Hund",
  "alter": 5,
  "spielzeug": ["Ball", "Seil"],
  "geimpft": true,
  "besitzer": null
}

In [4]:
# Define a model for a university course
# title: str          -> Attribute 'title' of type String
# professor: str       -> Attribute 'professor' of type String
# credits: int          -> Attribute 'credits' of type Integer
# semester: str | None -> Optional attribute 'semester', can be String or None
# BaseModel from Pydantic automatically generates the constructor, validation and attribute access
class Course(BaseModel):
    title: str
    professor: str
    credits: int
    semester: str | None

# Define a model for a list of courses
# courses: list[Course] -> Attribute 'courses' is a list of Course objects
class CourseList(BaseModel):
    courses: list[Course]

# We generate the JSON schema and include it in the prompt, so the model
# isn't left guessing what each field means - just relying on response_format
# alone only constrains the *shape* of the output, not its meaning.
json_schema = CourseList.model_json_schema()

prompt = f"""
I'm enrolled in two courses this semester.
"Introduction to Machine Learning" is taught by Prof. Müller and worth 6 credits. It's offered in the winter semester.
I'm also taking "Database Systems", a 5-credit course taught by Prof. Schmidt.

Extract this as JSON that strictly conforms to this schema:
{json.dumps(json_schema, indent=2)}
"""

completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_REASONING,
    reasoning_effort="none",  # not needed for a task this simple
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_format=CourseList,
)

output = completion.choices[0].message.content

# The LLM response is in string format. We need to convert it to a Python dictionary.
output_dict = json.loads(output)
print(output_dict)

{'courses': [{'title': 'Introduction to Machine Learning', 'professor': 'Prof. Müller', 'credits': 6, 'semester': 'winter semester'}, {'title': 'Database Systems', 'professor': 'Prof. Schmidt', 'credits': 5, 'semester': None}]}


### Example 2: Text Classification

We can also define a simple classification schema to categorize texts. In this example, we classify short texts into the categories "positive", "negative" or "neutral".

In [5]:
class SentimentResult(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"] # Literal defines allowed values for the attribute 'sentiment'

json_schema = SentimentResult.model_json_schema()

prompt = f"""Classify the sentiment of this text as positive, neutral, or negative:

I love programming in Python!

Return the result as JSON that strictly conforms to this schema:
{json.dumps(json_schema, indent=2)}
"""

completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_REASONING,
    reasoning_effort="none",  # not needed for a task this simple
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_format=SentimentResult,
)

output = completion.choices[0].message.content

# The LLM response is in string format, since LLMs output text. We need to convert the response to a Python dictionary.
output_dict = json.loads(output)
print(f"Output Dict: {output_dict}")

# Let's output only the sentiment
print(f"Sentiment: {output_dict['sentiment']}")

Output Dict: {'sentiment': 'positive'}
Sentiment: positive


### Example 3: JSON Schema in the Prompt

You've already seen the pattern above: pass the JSON schema directly to the LLM in the prompt, alongside `response_format`. `json.dumps(json_schema, indent=2)` converts the Python dictionary into a JSON string that can be used in the prompt. `indent=2` ensures readable formatting with indentation. 2 means that each level in the JSON is indented by 2 spaces.

This example applies the same pattern to a larger, more deeply nested schema, to show it scales beyond trivial cases.

In [6]:
# Define a Pydantic class for a single software developer
class Developer(BaseModel):
    name: str  # Name of the developer
    programming_language: Literal["python", "java", "javascript", "csharp", "cpp", "go", "rust", "php"] 
    experience_years: int = Field(ge=0, le=50)  # The developer has between 0 and 50 years of experience. Field enables validations
    specialization: Literal["frontend", "backend", "fullstack", "data_science", "devops", "mobile"]

# Define a Pydantic class for a list of exactly 2 developers
class DeveloperList(BaseModel):
    developers: List[Developer] = Field(min_length=2, max_length=2)  # Exactly 2 Developer objects

# Generate the JSON schema from the DeveloperList class
json_schema = DeveloperList.model_json_schema()

prompt = f"""
Generate a JSON array of exactly 2 software developers with the following fields:
- name: string
- programming_language: one of ["python", "java", "javascript", "csharp", "cpp", "go", "rust", "php"]
- experience_years: integer between 0 and 50
- specialization: one of ["frontend", "backend", "fullstack", "data_science", "devops", "mobile"]      
Make sure the output strictly adheres to this JSON schema:
{json.dumps(json_schema, indent=2)}
"""

completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_REASONING,
    reasoning_effort="none",  # not needed for a task this simple
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_format=DeveloperList,
)

output = completion.choices[0].message.content

# The LLM response is in string format. We need to convert it to a Python dictionary.
output_dict = json.loads(output)
print(output_dict)

{'developers': [{'name': 'Alice Henderson', 'programming_language': 'python', 'experience_years': 5, 'specialization': 'data_science'}, {'name': 'Marcus Thorne', 'programming_language': 'rust', 'experience_years': 12, 'specialization': 'backend'}]}


### Exercise 1

To practice validating JSON outputs, create a prompt that instructs the LLM to deliver a structured response in JSON format.

---

#### 🎯 Task

Define a simple JSON schema to generate dummy data for **researchers**.

#### 🎓 Researcher Schema

| Field          | Description                                                          |
| -------------- | --------------------------------------------------------------------- |
| `name`         | Name of the researcher                                                |
| `age`          | Age (between 25 and 80)                                               |
| `field`        | Research field from: **Computer Science**, **Biology**, **Physics**, **Mathematics** |
| `publications` | List of publications                                                   |

#### 📄 Publication Schema

| Field      | Description                          |
| ---------- | ------------------------------------- |
| `title`    | Publication title                     |
| `year`     | Publication year (between 1900 and 2023) |
| `keywords` | List of keywords                      |

---

#### 💡 Tip

Use Pydantic classes with `Field()` validations and the `Literal` type system for field selection! Remember to also include the generated JSON schema in the prompt.

<details>
<summary><b>Show Solution</b></summary>

```python
# Define a Pydantic class for a publication
class Publication(BaseModel):
    title: str
    year: int = Field(ge=1900, le=2023)
    keywords: List[str]

# Define a Pydantic class for a researcher
class Researcher(BaseModel):
    name: str
    age: int = Field(ge=25, le=80)
    field: Literal["Computer Science", "Biology", "Physics", "Mathematics"]
    publications: List[Publication]

# Define a Pydantic class for a list of researchers
class ResearcherList(BaseModel):
    researchers: List[Researcher] = Field(min_length=2, max_length=2)

# Generate the JSON schema from the ResearcherList class
json_schema = ResearcherList.model_json_schema()

prompt = f"""
Generate a list of 2 researchers with the following fields:
- name: string
- age: integer between 25 and 80
- field: one of ["Computer Science", "Biology", "Physics", "Mathematics"]
- publications: list of publications, each publication has:
  - title: string
  - year: integer between 1900 and 2023
  - keywords: list of keywords (strings)

Return the data as JSON that strictly conforms to this schema:
{json.dumps(json_schema, indent=2)}
"""

completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_REASONING,
    reasoning_effort="none",  # not needed for a task this simple
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_format=ResearcherList,
)

output = completion.choices[0].message.content

# The LLM response is in string format. We need to convert it to a Python dictionary.
output_dict = json.loads(output)
print(json.dumps(output_dict, indent=2, ensure_ascii=False))
```

</details>

In [7]:
### Your code...

### Example 4: Regex-Constrained Output (Beyond JSON)

So far, every example constrained the model's output to a **JSON schema**. But Ollama's native API also supports constraining raw output to match a **regex pattern** instead — useful when you want plain text in an exact shape, without going through JSON at all. This is done via the `format` field of Ollama's native `/api/chat` endpoint (not the OpenAI-compatible one), so here we call it directly with the `requests` library instead of the `openai` package.

In [8]:
import requests

data = {
    "model": LLM_REASONING,
    "messages": [{"role": "user", "content": "Create a name and age for a random person."}],
    "stream": False,
    "think": False,  # not needed for a task this simple
    "format": {"type": "string", "pattern": "^Name: [A-Za-z ]+\nAge: [0-9]+$"},
}

response = requests.post(f"{OLLAMA_HOST}/api/chat", json=data)
print(response.json()["message"]["content"])

"Name: Elena Vance  
Age: 34"


One important gotcha we ran into while preparing this notebook: a pattern like `^Name: .*\nAge: .*$` — using the unbounded wildcard `.*`, which is what you'd naturally reach for — causes Ollama's regex-to-grammar compiler to hang for minutes rather than returning a result. This reproduced across every model we tried (`gemma3:4b`, `gemma3:27b`, and `gemma4:26b`), not just `LLM_REASONING`, so it's a general limitation of the regex-format feature itself. Prefer bounded character classes like `[A-Za-z ]+` or `[0-9]+` over `.*` whenever you use regex-constrained output — they compile essentially instantly, as you saw above.

#### Does this also work with the `openai` package?

No — not reliably. The `openai` package (and any raw request to Ollama's OpenAI-compatible `/v1/chat/completions` endpoint) doesn't honor the native `format` regex field at all: it's silently ignored, and you just get an unconstrained response back, with no error to warn you. Regex-constrained output is only available through Ollama's native `/api/chat` endpoint. See for yourself — same `format` field, but sent through the `openai` client this time:

In [9]:
completion = client.chat.completions.create(
    model=LLM_REASONING,
    messages=[{"role": "user", "content": "Create a name and age for a random person."}],
    temperature=0,
    extra_body={"format": {"type": "string", "pattern": "^Name: [A-Za-z ]+\nAge: [0-9]+$"}},
)

print(completion.choices[0].message.content)  # not constrained - the format field was silently ignored

**Name:** Elias Thorne
**Age:** 34


#### Tip: describe the format in the prompt too

Even when using a grammar/schema constraint like this — or **guided JSON**, as in the other examples in this notebook — it's still good practice to also describe the desired format directly in the prompt text (like we did in Example 3, by embedding the JSON schema in the prompt itself). The constraint only guarantees the output is *syntactically* valid; it doesn't guarantee the model picks sensible content, or fully understands what you actually want in each field. Explicitly describing the format helps guide *what* the model writes, not just how it's shaped.

### Example 5: Guided JSON for Structured Extraction of Information from Documents

Let's first check whether the model can read an image at all, before adding any structure.

Unlike the simpler examples above, we'll leave `LLM_REASONING`'s thinking **enabled** for this section — extracting many fields from a real document is exactly the kind of harder task where reasoning can genuinely help accuracy, so the extra time is worth it here.

Example:

<img src="content/billing.png" alt="Purchase Order Example" width="500">

In [10]:
# Let's load the image and encode it in Base64. Base64 is a text format that converts binary data (like images) into a text representation.
import base64
with open("content/billing.png", "rb") as image_file:
    # Convert file to Base64
    encoded_string = base64.b64encode(image_file.read()).decode("utf-8")

# Format as data URL. Data URLs allow embedding images directly in HTML or JSON.
data_url = f"data:image/png;base64,{encoded_string}"

In [11]:
# Let's start without JSON validation to test the output.
# The LLM should first be able to analyze an image and generate a response.
prompt = "What do you see in this image?"
completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_REASONING,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {
                    "type": "image_url",
                    "image_url": data_url
                },
            ],
        }
    ]
)

print(completion.choices[0].message.content)

This image is a template for a **Purchase Order** from "BIZZLIBRARY.COM". It contains several structured sections:

*   **Header Information:** Includes placeholders for company name, address, phone number, email, and website. It also features a placeholder for the date (`mm/dd/yyyy`) and a specific purchase order number (`8365234`).
*   **Vendor Information:** Lists "ABC Office Supplies" located in Washington DC as the vendor, with John Smith listed as the sales person.
*   **Customer Information:** Identifies "Franklin Middle School" in Washington DC as the customer, with Helen Wilson from the Purchasing Department as the contact person.
*   **Itemized List:** A table detailing six different office supply items:
    1.  Pencils HB (5 dozen) - $50.00
    2.  Pencils 2B (4 dozen) - $40.00
    3.  Paper - A4, Photo copier, 70 gram (10 reams) - $30.00
    4.  Paper - A4, Photo copier, 80 gram (15 reams) - $48.00
    5.  Pen - Ball Point, Blue (10 boxes) - $100.00
    6.  Highlighter - 3 

#### Now With Guided JSON: Extracting Receipt Details

Now we will look at a more complex example: structured extraction of information from a receipt.

#### Goal of the Exercise

We want to analyze a restaurant receipt and automatically extract the most important information:
- **List of items** ordered with details
- **Total amount** of the bill

#### 🖼️ The Document

This is what the receipt to be analyzed looks like:

<img src="content/receipt.png" alt="Receipt Example" width="400">

In [12]:
with open("content/receipt.png", "rb") as image_file:
    encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
data_url = f"data:image/png;base64,{encoded_string}"

In [13]:
class ReceiptItem(BaseModel):
    # Note: deliberately no max_length here - a str max_length constraint nested
    # inside a list item's model breaks LLM_REASONING's grammar compiler ("failed
    # to parse grammar"). min_length alone is fine; it's specifically max_length
    # on a nested string field that causes it.
    description: str = Field(min_length=1)
    quantity: int
    price_usd: float

class Receipt(BaseModel):
    total_amount: float = Field(gt=0)
    items: List[ReceiptItem] = Field(min_length=1)

In [14]:
json_schema = Receipt.model_json_schema()

prompt = f"""Analyze the receipt in the image and give me the details as JSON. Extract all items with their descriptions, quantities, and prices.

Return the data as JSON that strictly conforms to this schema:
{json.dumps(json_schema, indent=2)}
"""
completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_REASONING,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                },
                {
                    "type": "image_url",
                    "image_url": data_url
                },
            ],
        }
    ],
    response_format=Receipt,
)

print(completion.choices[0].message.content)

{
  "total_amount": 157.51,
  "items": [
    {
      "description": "Shrimp Scampi",
      "quantity": 1,
      "price_usd": 27.0
    },
    {
      "description": "Chicken Milanese",
      "quantity": 2,
      "price_usd": 46.0
    },
    {
      "description": "Veal Milanese",
      "quantity": 1,
      "price_usd": 27.0
    },
    {
      "description": "Grey Goose",
      "quantity": 3,
      "price_usd": 30.0
    },
    {
      "description": "Pineapple Juice",
      "quantity": 1,
      "price_usd": 3.0
    },
    {
      "description": "Belvedere",
      "quantity": 1,
      "price_usd": 12.0
    },
    {
      "description": "Becks Non Alcoholic",
      "quantity": 1,
      "price_usd": 6.0
    },
    {
      "description": "Corona",
      "quantity": 1,
      "price_usd": 6.0
    },
    {
      "description": "Comp Item",
      "quantity": 1,
      "price_usd": -12.0
    }
  ]
}


### Exercise 2

Now it's your turn! You want to automatically extract information from the purchase order. 

Example:

<img src="content/billing.png" alt="Purchase Order Example" width="500">

The LLM should return the following structured information in JSON format:

* Vendor information:
  - Vendor name
  - Address
  - Contact number
  - Email address

* Customer information:
  - Customer name
  - Address
  - Contact number
  - Email address

  - Total amount

* Purchase order details:  - Discount percentage and amount

  - Purchase order number  - Tax percentage and amount

  - Date  - Subtotal

* Financial summary:

* List of items, each with:

  - Item description  - Total price

  - Unit  - Unit price
  - Quantity

<details>
<summary><b>Show Solution</b></summary>

```python
# Define a Pydantic class for contact information
class ContactInfo(BaseModel):
    name: str
    address: str
    contact_number: Optional[str] = None
    email_address: Optional[str] = None

# Define a Pydantic class for an order item
class OrderItem(BaseModel):
    description: str
    unit: str
    quantity: int = Field(ge=0)
    unit_price: float = Field(ge=0)
    total_price: float = Field(ge=0)

# Define a Pydantic class for financial summary
class FinancialSummary(BaseModel):
    subtotal: float = Field(ge=0)
    tax_percentage: float = Field(ge=0, le=100)
    tax_amount: float = Field(ge=0)
    discount_percentage: float = Field(ge=0, le=100)
    discount_amount: float = Field(ge=0)
    total: float = Field(ge=0)

# Define a Pydantic class for the purchase order
class PurchaseOrder(BaseModel):
    vendor: ContactInfo
    customer: ContactInfo
    purchase_order_number: str
    date: str
    items: List[OrderItem] = Field(min_length=1)
    financial_summary: FinancialSummary

# Let's load the image and encode it in Base64
with open("content/billing.png", "rb") as image_file:
    encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
data_url = f"data:image/png;base64,{encoded_string}"

json_schema = PurchaseOrder.model_json_schema()

prompt = f"""Analyze the purchase order in the image and extract all information including vendor details, customer details, order items, and financial summary as JSON.

Return the data as JSON that strictly conforms to this schema:
{json.dumps(json_schema, indent=2)}
"""

completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_REASONING,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {
                    "type": "image_url",
                    "image_url": data_url
                },

            ],
        }
    ],
    response_format=PurchaseOrder
)

# The LLM response is in string format. We need to convert it to a Python dictionary.
output = completion.choices[0].message.content
output_dict = json.loads(output)
print(json.dumps(output_dict, indent=2, ensure_ascii=False))
```

</details>

In [15]:
# Your code goes here...